# Неделя 3 — Baseline-модели

Задание: `docs/week3_baseline.md`

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append('..')
from src.validation import (TEST_SNAPSHOTS, TRAIN_SNAPSHOTS, VALID_SNAPSHOTS,
                            evaluate, split_by_time)

dataset = pd.read_parquet('../data/processed/dataset.parquet')
print(dataset.shape)

(257631, 27)


## 1. Разбиение по времени (out-of-time)

In [2]:
train, valid, test = split_by_time(dataset)
print(len(train), len(valid), len(test))
print('доля оттока:', train['target'].mean().round(4), test['target'].mean().round(4))

feature_cols = [
    col for col in dataset.columns
    if col not in ["client_id", "snapshot", "target"]
]

y_train = train['target']
x_train = train[feature_cols]

x_valid = valid[feature_cols]
y_valid = valid["target"]

x_test = test[feature_cols]
y_test = test['target']

# print(x_train.shape, y_train.shape)
# print(x_test.shape, y_test.shape)

66253 35788 40265
доля оттока: 0.0762 0.1379


## 2. Baseline 0 — правило

In [3]:
rule_valid_metrics = evaluate(y_valid, -x_valid['rev_last_to_mean'], 'Правило: падение платежа')

# rule_valid_metrics = evaluate(y_test, -x_test['rev_last_to_mean'], 'Правило: падение платежа')


--- Правило: падение платежа ---
ROC-AUC: 0.8618
PR-AUC: 0.6538
Precision@10%: 0.6417
Lift@10%: 6.93x
Доля оттока: 0.0925


## 3. Baseline 1 — логистическая регрессия

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = x_train.select_dtypes(
    include = ["number"],
).columns.tolist()

categorical_cols = x_train.select_dtypes(
    include = ["object","string","category"],
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
     ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preproccesor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

logreg = Pipeline([
    ("preprocessor", preproccesor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])
print(f"Числовые признаки ({len(numeric_cols)}): {numeric_cols}")
print(f"Категориальные признаки ({len(categorical_cols)}): {categorical_cols}")

logreg.fit(x_train, y_train)
valid_score = logreg.predict_proba(x_valid)[:, 1]

logreg_valid_metrics = evaluate(
    y_valid, 
    valid_score, 
    "Baseline 1"
)

Числовые признаки (21): ['rev_mean', 'rev_min', 'rev_max', 'rev_last', 'traffic_mean', 'traffic_last', 'sim_last', 'sim_mean', 'sum_tickets', 'last_tickets', 'sum_debt', 'last_debt', 'max_debt', 'month_in_obs', 'rev_last_to_mean', 'rev_std', 'rev_trend', 'traffic_trend', 'rev_trend_norm', 'rev_months_declining', 'sim_change']
Категориальные признаки (3): ['segment', 'product', 'region']
--- Baseline 1 ---
ROC-AUC: 0.8786
PR-AUC: 0.7117
Precision@10%: 0.6453
Lift@10%: 6.97x
Доля оттока: 0.0925


## 4. Сравнение baseline'ов

TODO: таблица метрик правило vs логрегрессия.

In [6]:
comparison = pd.DataFrame([
    {
        "Модель": "Правило",
        "ROC-AUC": 0.8618,
        "PR-AUC": 0.6538,
        "Precision@10%": 0.6417,
        "Lift@10%": 6.93,
        "Доля оттока": 0.0925 
        
    },
    {
        "Модель": "Логистическая регрессия",
        "ROC-AUC": 0.8786,
        "PR-AUC": 0.7117,
        "Precision@10%": 0.6453,
        "Lift@10%": 6.97,
        "Доля оттока": 0.0925
    }
])

comparison.round(4)

,Модель,ROC-AUC,PR-AUC,Precision@10%,Lift@10%,Доля оттока
0,Правило,0.8618,0.6538,0.6417,6.93,0.0925
1,Логистическая регрессия,0.8786,0.7117,0.6453,6.97,0.0925
